# Lab 8: Distilling a vision model

The method from Modules 2, 3 and 5, with photographs. The task: name the breed of a cat or dog,
out of 37.

1. Build the four piles of images.
2. Score every photo with a teacher that needs no training, and save the scores.
3. Baseline: a small student trained on 148 labelled photos.
4. A small student distilled from the teacher's probabilities.
5. A larger student distilled with LoRA.
6. Compare, and look at some mistakes.

This lab stands alone. Allow about 40 minutes. The first run downloads about 800 MB of images.

In [ ]:
from kd_course.env import check_environment

report = check_environment()
print(report.summary())
assert report.ok, "Fix the problems listed above before continuing."

## 1. The data

In [ ]:
from kd_course.vision import load_pets, make_image_splits

bundle = load_pets()
splits = make_image_splits(bundle.train, bundle.test)
LABELS = bundle.label_names
N = len(LABELS)

print(f"Breeds:              {N}")
print(f"Gold training set:   {len(splits.gold_train)}")
print(f"Gold validation set: {len(splits.gold_val)}")
print(f"Unlabelled pool:     {len(splits.pool)}")
print(f"Test set:            {len(splits.test)}")
splits.gold_train.images[0].resize((224, 224))

## 2. The teacher

It turns each breed name into a sentence and scores how well each photo matches each sentence.
Scores are saved; running this cell again reads them back.

In [ ]:
from kd_course.label_scoring import top_labels
from kd_course.metrics import summarize
from kd_course.results import default_results_dir, save_result
from kd_course.vision import load_zero_shot_teacher, zero_shot_or_load

results_dir = default_results_dir() / "vision"
scores_dir = results_dir / "teacher_scores"

teacher, teacher_processor, teacher_tokenizer = load_zero_shot_teacher()
args = {
    "model": teacher,
    "image_processor": teacher_processor,
    "tokenizer": teacher_tokenizer,
    "label_names": LABELS,
    "progress_every": 500,
}
test_scores = zero_shot_or_load(scores_dir / "test.json", splits.test, **args)
gold_scores = zero_shot_or_load(scores_dir / "gold_train.json", splits.gold_train, **args)
pool_scores = zero_shot_or_load(scores_dir / "pool.json", splits.pool, **args)

teacher_pred = top_labels(test_scores)
teacher_result = summarize(splits.test.labels, teacher_pred, N)
save_result("teacher_zero_shot", teacher_result, teacher_pred, results_dir)
print(teacher_result)

In [ ]:
import gc

import torch


def free():
    """Release GPU memory. Delete your own references to a model before calling this."""
    gc.collect()
    torch.cuda.empty_cache()


del teacher, args
free()

## 3. Prepare the images once

Each image is resized and converted to numbers a single time, then reused in every epoch.

In [ ]:
from transformers import AutoImageProcessor

from kd_course.models import resolve_model_path
from kd_course.vision import VISION_STUDENT_ID, VISION_STUDENT_LARGE_ID, to_pixels


def prepare(model_id):
    processor = AutoImageProcessor.from_pretrained(resolve_model_path(model_id))
    return {
        "gold": to_pixels(processor, splits.gold_train.images),
        "val": to_pixels(processor, splits.gold_val.images),
        "pool": to_pixels(processor, splits.pool.images),
        "test": to_pixels(processor, splits.test.images),
    }


small = prepare(VISION_STUDENT_ID)
print({name: tuple(tensor.shape) for name, tensor in small.items()})

## 4. Baseline

The small student, trained on the 148 gold photos and nothing else.

In [ ]:
from kd_course.vision import predict_images, train_image_classifier

predictions = {}


def score(name, student, pixels):
    predictions[name] = predict_images(student, pixels["test"])
    result = summarize(splits.test.labels, predictions[name], N)
    save_result(name, result, predictions[name], results_dir)
    print(name, result)
    return result


baseline = train_image_classifier(
    VISION_STUDENT_ID,
    small["gold"],
    splits.gold_train.labels,
    small["val"],
    splits.gold_val.labels,
    num_labels=N,
    epochs=20,
)
baseline_result = score("baseline_gold_148", baseline, small)
del baseline
free()

## 5. The distilled student

Same model. It also learns from the teacher's probabilities for the pool. Pool photos have no
human label (`None`). The loss is `kd_loss` from Module 3.

In [ ]:
mixed_labels = splits.gold_train.labels + [None] * len(splits.pool)
teacher_scores = gold_scores + pool_scores

distilled = train_image_classifier(
    VISION_STUDENT_ID,
    torch.cat([small["gold"], small["pool"]]),
    mixed_labels,
    small["val"],
    splits.gold_val.labels,
    num_labels=N,
    teacher_log_probs=teacher_scores,
    temperature=2.0,
    alpha=0.5,
    epochs=5,
)
distilled_result = score("distilled_small", distilled, small)
del distilled
free()

## 6. A larger student with LoRA

About fifteen times larger. Its attention layers get adapters, its new output layer trains in full,
and everything else is frozen.

In [ ]:
from kd_course.lora import count_parameters_by_kind

large = prepare(VISION_STUDENT_LARGE_ID)
lora = train_image_classifier(
    VISION_STUDENT_LARGE_ID,
    torch.cat([large["gold"], large["pool"]]),
    mixed_labels,
    large["val"],
    splits.gold_val.labels,
    num_labels=N,
    teacher_log_probs=teacher_scores,
    temperature=2.0,
    alpha=0.5,
    epochs=5,
    learning_rate=3e-4,
    lora_rank=16,
)
counts = count_parameters_by_kind(lora.model)
share = counts["trainable_share"]
print(f"Trained {counts['trainable']:,} of {counts['total']:,} weights ({share:.2%}).")
lora_result = score("distilled_large_lora", lora, large)
del lora, large
free()

## 7. Compare

In [ ]:
from kd_course.metrics import accuracy, gap_closed, paired_bootstrap
from kd_course.results import format_results, load_results

print(format_results(load_results(results_dir)))
print()
for name, result in (("Small distilled", distilled_result), ("Large with LoRA", lora_result)):
    gap = gap_closed(baseline_result["accuracy"], result["accuracy"], teacher_result["accuracy"])
    print(f"{name:<16}", "no gap to close" if gap is None else f"{gap:.0%} of the gap closed")


def report(title, a, b):
    r = paired_bootstrap(accuracy, splits.test.labels, predictions[a], predictions[b])
    print(
        f"{title}: {r.difference * 100:+.1f} points "
        f"(95% interval {r.low * 100:+.1f} to {r.high * 100:+.1f})"
    )


print()
report("Distilled minus baseline      ", "baseline_gold_148", "distilled_small")
report("Large LoRA minus small distilled", "distilled_small", "distilled_large_lora")

## 8. Look at some mistakes

Six test photos the small distilled student got wrong, with its answer, the truth, and the
teacher's top choice.

In [ ]:
from IPython.display import display

wrong = [
    i
    for i, (p, t) in enumerate(zip(predictions["distilled_small"], splits.test.labels, strict=True))
    if p != t
]
for i in wrong[:6]:
    display(splits.test.images[i].resize((160, 160)))
    print(
        f"student: {LABELS[predictions['distilled_small'][i]]}   "
        f"truth: {LABELS[splits.test.labels[i]]}   teacher: {LABELS[teacher_pred[i]]}"
    )

## 9. What did you see?

Double-click this cell and write your answers.

**a. How far apart were the teacher and the baseline? Compare that with your Lab 2 result on the banking task. Why is it different here?**

**b. How much of the gap did the small distilled student close? Is its gain over the baseline real? Quote the interval.**

**c. The larger student differs from the small one in two ways. Which do you think explains most of the difference in their scores, and how would you test that?**

**d. In the mistakes you looked at, how often did the teacher make the same mistake as the student? What does that suggest?**

**e. Nothing in `kd_loss` changed between Module 3 and this lab. What does that tell you about where the method's generality comes from?**